<a href="https://colab.research.google.com/github/iabbey-commits/Financial-Research-Agent/blob/main/Agent4_RAG_Evaluator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**AI-Agent 4**: LLM As An Evaluator.

In this errar of technology, information has been made available to all but the authenticity of the  information is what matters most.
This agent has been built to evaluate and validate information relating to financial report analysis. The agent uses RAG system to retrieve information from a source (LLM) upon a user's request. The RAG Evaluator class then sends an evaluation prompt the Agno agent (LLM wrapper) which process it and forward it to the Groq LLM. This then score or assess the perfomance of the RAG system using an evaluation engine to come out with a valid information.

The main aim of the agent is to automatically assess the performance of RAG systems  without requiring constant human assessment.The agent helps to determine whether the RAG system retrieves relevant information, avoids hallucination, uses the retrieved context properly, and produces coherent, relevant, and complete responses. The agent functions as the validation and quality assurance layer, ensuring the reliability of the system.


Below is the Architecture diagram for the workflow of the agent.

In [ ]:
from graphviz import Digraph

def create_rag_evaluator_architecture():
    agent4 = Digraph(format='png')
    agent4.attr(rankdir='LR', size='20,15')

    # Global node style
    agent4.attr('node', shape='rectangle', style='filled', color='lightblue', fontname='Helvetica')

    # Nodes
    agent4.node('A', 'User Query')

    agent4.node('B', '''RAG System
- Retriever
- Knowledge Base
- Generator (LLM)''')

    agent4.node('C', 'RAGEvaluator Class\n(Prompt Builder)')

    agent4.node('D', '''Agno Agent
(LLM Wrapper)''')

    agent4.node('E', '''Groq LLM
Llama 3.1 8B
(LLM-as-a-Judge)''')

    agent4.node('F', 'Evaluation Engine', shape='circle', style='filled', fillcolor='yellow')
    agent4.node('F1', 'Faithfulness Score', shape='box', style='filled', fillcolor='blue')
    agent4.node('F2', 'Context Relevance', shape='box', style='filled', fillcolor='green')
    agent4.node('F3', 'Answer Completeness', shape='box', style='filled', fillcolor='lightgreen')
    agent4.node('F4', 'Source Attribution', shape='box', style='filled', fillcolor='lightblue')
    agent4.node('F5', 'Response Coherence', shape='box', style='filled', fillcolor='magenta')

    agent4.node('G', '''Evaluation Report
(Structured Output)''', shape='circle', style='filled', fillcolor='orange')

    # Edges
    agent4.edge('A', 'B', label='Query')
    agent4.edge('B', 'C', label='Context + Response')
    agent4.edge('C', 'D', label='Evaluation Prompt')
    agent4.edge('D', 'E', label='Processed Prompt')
    agent4.edge('E', 'F', label='Scoring')
    agent4.edge('F', 'F1') #label='Final Output')
    agent4.edge('F', 'F2') #label='Final Output')
    agent4.edge('F', 'F3') #label='Final Output')
    agent4.edge('F', 'F4') #label='Final Output')
    agent4.edge('F', 'F5') #label='Final Output')
    agent4.edge('F1', 'G', label='Final Output')
    agent4.edge('F2', 'G', label='Final Output')
    agent4.edge('F3', 'G', label='Final Output')
    agent4.edge('F4', 'G', label='Final Output')
    agent4.edge('F5', 'G', label='Final Output')

    return agent4


# Generate and save diagram
diagram = create_rag_evaluator_architecture()
diagram.render('rag_evaluator_architecture', view=True)

In [ ]:
from IPython.display import Image
Image('rag_evaluator_architecture.png')

Below is the code for the agent.

In [ ]:
!pip install agno groq


In [ ]:
import os
from google.colab import userdata
userdata.get('GROQ_API_KEY')

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
#os.environ['PHI_API_KEY'] = userdata.get('PHI_API_KEY') #INPUT YOUR AGNO KEY (I had this key before the rebrand)

print("API keys have been set!")


In [ ]:
from textwrap import dedent
from agno.agent import Agent
from agno.models.groq import Groq

class RAGEvaluator:
    def __init__(self):
        self.evaluator = self._initialize_evaluator()

    def _initialize_evaluator(self):
        return Agent(
            model=Groq(id="openai/gpt-oss-120b"),  # Using different Llama model
            description=dedent("""\
                You are an expert RAG system evaluator with deep expertise in:
                - Information retrieval quality assessment
                - Response accuracy evaluation
                - Source attribution verification
                - Context relevance analysis
                - Natural language generation evaluation
            """),
            instructions=dedent("""\
                Evaluate the RAG system output based on these key metrics:

                1. Faithfulness (1-5):
                   - How accurately does the response reflect the source documents?
                   - Are there any hallucinations or incorrect statements?
                   - Does it maintain factual consistency?

                2. Context Relevance (1-5):
                   - Are the retrieved passages relevant to the query?
                   - Is important context missing?
                   - Is irrelevant information included?

                3. Answer Completeness (1-5):
                   - Does the response fully address the query?
                   - Are all key aspects covered?
                   - Is the level of detail appropriate?

                4. Source Attribution (1-5):
                   - Are sources properly cited?
                   - Is it clear which information comes from where?
                   - Can claims be traced back to sources?

                5. Response Coherence (1-5):
                   - Is the response well-structured?
                   - Does it flow logically?
                   - Is it easy to understand?

                Provide specific examples and explanations for each score.
            """),
            expected_output=dedent("""\
                # RAG Evaluation Report

                ## Overview
                Query: {query}
                Response Length: {n_chars} characters

                ## Metric Scores

                ### Faithfulness: {score}/5
                - Justification:
                - Examples:
                - Areas for Improvement:

                ### Context Relevance: {score}/5
                - Justification:
                - Examples:
                - Areas for Improvement:

                ### Answer Completeness: {score}/5
                - Justification:
                - Examples:
                - Areas for Improvement:

                ### Source Attribution: {score}/5
                - Justification:
                - Examples:
                - Areas for Improvement:

                ### Response Coherence: {score}/5
                - Justification:
                - Examples:
                - Areas for Improvement:

                ## Overall Score: {total}/25

                ## Key Recommendations
                1. {rec1}
                2. {rec2}
                3. {rec3}

                ## Summary
                {final_assessment}
            """),
            markdown=True,
        )

    def evaluate(self, query: str, response: str, context: list, stream: bool = True):
        """
        Evaluate a RAG system's response

        Args:
            query (str): Original user query
            response (str): RAG system's response
            context (list): Retrieved passages used for the response
            stream (bool): Whether to stream the evaluation output
        """
        evaluation_prompt = f"""
        Please evaluate this RAG system output:

        QUERY:
        {query}

        RETRIEVED CONTEXT:
        {' '.join(context)}

        RESPONSE:
        {response}

        Provide a detailed evaluation following the metrics and format specified.
        """

        return self.evaluator.print_response(evaluation_prompt, stream=stream)


# Initialize evaluator
evaluator = RAGEvaluator()
print("LLM-as-a Judge Evaluator initialized successfully!")

In [ ]:
# Example evaluation. Rerun this to use actual financial RAG outputs

query = "What are the key features of transformer models?"
context = [
    "Transformer models use self-attention mechanisms to process input sequences.",
    "Key features include parallel processing and handling of long-range dependencies."
]
response = "Transformer models are characterized by their self-attention mechanism..."

# Run evaluation
evaluator.evaluate(query, response, context)